In [1]:
from typing import List, Optional

class Node:
    def __init__(self, is_leaf: bool):
        self.is_leaf = is_leaf
        self.parent: Optional['InternalNode'] = None

class LeafNode(Node):
    def __init__(self):
        super().__init__(is_leaf=True)
        self.keys: List[int] = []

class InternalNode(Node):
    def __init__(self):
        super().__init__(is_leaf=False)
        self.keys: List[int] = []
        self.children: List[Node] = []


#bplus tree class
class BPlusTree:

    def __init__(self, d: int):
        self.d =d
        self.root: Node = LeafNode() # in the beginning, root is a leaf

    #! i use this to find leaf node during insertion and deletion
    # where it should get placed or deleted
    def get_leaf(self, key: int) -> LeafNode:
        node = self.root
        while not node.is_leaf:
            inode: InternalNode = node  
            i = 0
            while i<len(inode.keys) and key>=inode.keys[i]:
                i+=1
            node=inode.children[i]
        return node  

#*------------------------------------------------------------------------------------------------------------------------


    #! split and COPY UP
    def split_leaf(self, leaf: LeafNode):
        total = len(leaf.keys)

        left_count = self.d 
        right_keys = leaf.keys[left_count:]
        left_keys = leaf.keys[:left_count] #! when splitting, left gets d keys, right gets d+1 keys (more)

        new_leaf = LeafNode()
        new_leaf.keys = right_keys
        leaf.keys = left_keys

        # parent handling
        key_to_parent = new_leaf.keys[0]
        parent = leaf.parent

        if parent is None:
            # create new root
            new_root = InternalNode()
            new_root.keys = [key_to_parent]
            new_root.children = [leaf, new_leaf]
            leaf.parent = new_root
            new_leaf.parent = new_root
            self.root = new_root

        else:
            # insert right node into the parent of left node
            #! case 1) no parent, if the split leaf was the root, then i will create a new roor
            if leaf.parent is None:
                new_root = InternalNode()
                new_root.keys = [key_to_parent]
                new_root.children = [leaf, new_leaf]
                leaf.parent = new_root
                new_leaf.parent = new_root
                self.root = new_root
                return

            #! case 2) if parent present, insert into parent
            parent: InternalNode = leaf.parent  
            idx = parent.children.index(leaf)
            parent.children.insert(idx+ 1, new_leaf)
            parent.keys.insert(idx, key_to_parent)
            new_leaf.parent = parent

            #! special case **) if parent is there and it overflows, split recursively again
            if len(parent.keys) > 2 * self.d:
                self.split_internal(parent)



    #! split and PUSH UP
    def split_internal(self, inode: InternalNode):
        total = len(inode.keys)

        mid_index = self.d  
        push_up_key = inode.keys[mid_index]

        left_keys = inode.keys[:mid_index]
        right_keys = inode.keys[mid_index+1:] 

        left_children = inode.children[:mid_index+1]
        right_children = inode.children[mid_index+1:]

        left_node = InternalNode()
        left_node.keys = left_keys
        left_node.children = left_children
        for c in left_children:
            c.parent = left_node

        right_node = InternalNode()
        right_node.keys = right_keys
        right_node.children = right_children
        for c in right_children:
            c.parent = right_node


        parent = inode.parent
        if parent is None:
            # create new root
            new_root = InternalNode()
            new_root.keys = [push_up_key]
            new_root.children = [left_node, right_node]
            left_node.parent = new_root
            right_node.parent = new_root
            self.root = new_root
        else:
            # PUSH UP
            idx = parent.children.index(inode)
            parent.children[idx] = left_node
            parent.children.insert(idx+1, right_node)
            parent.keys.insert(idx, push_up_key)
            left_node.parent = parent
            right_node.parent = parent
            
            #! case : recursive split
            if len(parent.keys) > 2 * self.d:
                self.split_internal(parent)



    def insert(self, key: int):

        leaf = self.get_leaf(key)

        import bisect
        pos = bisect.bisect_left(leaf.keys, key)
        
        # ignore if key already present
        if pos < len(leaf.keys) and leaf.keys[pos] == key:
            return
        
        #! insert key into leaf
        leaf.keys.insert(pos, key)
        
        # check overflow
        #! case: overflow (this will call only split_leaf, and if any internal node overflows,
        #! the split_leaf function internally calls split_internal)
        if len(leaf.keys) > 2 * self.d:
            self.split_leaf(leaf)

#*------------------------------------------------------------------------------------------------------------------------


    def _merge_leaves(self, left: LeafNode, right: LeafNode, parent:InternalNode, idx_in_parent:int):
        #! merge right into left
        left.keys.extend(right.keys)
        
        parent.children.pop(idx_in_parent+1) 
        parent.keys.pop(idx_in_parent)       
        # if parent becomes empty and is root, adjust root
        self.fix_internal_after_delete(parent)




    def fix_internal_after_delete(self, inode: InternalNode):
        
        #! case: internal node is root
        if inode.parent is None:
            if len(inode.keys) == 0:
                # shrink tree height: make single child the root
                child = inode.children[0]
                child.parent = None
                self.root = child
            return

        #! case: underflow
        if len(inode.keys) < self.d:
            # try to borrow from siblings (internal node borrow)
            parent = inode.parent
            idx = parent.children.index(inode)
            # try left sibling
            left_sib = None
            right_sib = None
            if idx - 1 >= 0:
                left_sib = parent.children[idx-1]
            if idx + 1 < len(parent.children):
                right_sib = parent.children[idx+1]

            #! if possible borrow from left sibling if it exists and has more than d keys
            if left_sib and not left_sib.is_leaf and len(left_sib.keys) > self.d:
                ls: InternalNode = left_sib  
                borrow_key = ls.keys.pop(-1)
                borrow_child = ls.children.pop(-1)
                inode.keys.insert(0, parent.keys[idx-1])
                parent.keys[idx-1] = borrow_key
                inode.children.insert(0, borrow_child)
                borrow_child.parent = inode
                return
            
            #!  borrow from right sibling
            if right_sib and not right_sib.is_leaf and len(right_sib.keys) > self.d:
                rs: InternalNode = right_sib  
                borrow_key = rs.keys.pop(0)
                borrow_child = rs.children.pop(0)
                inode.keys.append(parent.keys[idx])
                parent.keys[idx] = borrow_key
                inode.children.append(borrow_child)
                borrow_child.parent = inode
                return
            

            #! case: merge (when borrowing not possible)
            #! merge with left sibling
            if left_sib and not left_sib.is_leaf:
                ls: InternalNode = left_sib  
                
                sep = parent.keys.pop(idx-1)
                parent.children.pop(idx)  
                ls.keys.append(sep)
                ls.keys.extend(inode.keys)
                ls.children.extend(inode.children)
                for c in inode.children:
                    c.parent = ls
                
                #! after merge, fix parent (recursively)
                self.fix_internal_after_delete(parent)
                return
            

            #! merge right_sib into inode
            if right_sib and not right_sib.is_leaf:
                rs: InternalNode = right_sib  
                sep = parent.keys.pop(idx)
                parent.children.pop(idx+1)
                inode.keys.append(sep)
                inode.keys.extend(rs.keys)
                inode.children.extend(rs.children)
                for c in rs.children:
                    c.parent = inode
                self.fix_internal_after_delete(parent)
                return


    def delete(self, key: int):
            leaf = self.get_leaf(key)

            #! case: key not present
            if key not in leaf.keys: return  
            leaf.keys.remove(key)

            #! case: if leaf is root
            if leaf.parent is None:
                return

            #! case: no underflow
            if len(leaf.keys) >= self.d:
                parent = leaf.parent
                idx = parent.children.index(leaf)
                if idx > 0:
                    parent.keys[idx-1] = leaf.keys[0]
                return

            #! case: underflow (borrow)
            parent = leaf.parent
            idx = parent.children.index(leaf)
            left_sib = parent.children[idx-1] if idx-1 >= 0 else None
            right_sib = parent.children[idx+1] if idx+1 < len(parent.children) else None

            #! first, try to borrow from left sibling
            if left_sib and left_sib.is_leaf:
                if len(left_sib.keys) > self.d:
                    # borrow last key of left sibling
                    borrowed = left_sib.keys.pop(-1)
                    leaf.keys.insert(0, borrowed)
                    # update parent's separator key for these two children
                    # parent.keys[idx_in_parent] is separator (between left_sib and leaf)
                    parent.keys[idx-1] = leaf.keys[0]  # copy up first key of leaf
                    return
                
            #! then try to borrow from right sibling
            if right_sib and right_sib.is_leaf:
                if len(right_sib.keys) > self.d:
                    borrowed = right_sib.keys.pop(0)
                    leaf.keys.append(borrowed)
                    # parent key between leaf and right_sib should be updated to right_sib.first
                    parent.keys[idx] = right_sib.keys[0]
                    return


            #! case: merge (when borrowing not possible)
            #! prefer merging with left sibling if exists, else right sibling
            if left_sib and left_sib.is_leaf:
                self._merge_leaves(left_sib, leaf, parent, idx-1)
                return
            if right_sib and right_sib.is_leaf:
                self._merge_leaves(leaf, right_sib, parent, idx)
                return
    
#*------------------------------------------------------------------------------------------------------------------------

    def init_tree_from_structure(self, root_node: Node):
        self.root = root_node
        def set_parents(node, parent):
            node.parent = parent
            if node.is_leaf: return
            inode: InternalNode = node  
            for c in inode.children:
                set_parents(c, inode)

        set_parents(self.root, None)

#*------------------------------------------------------------------------------------------------------------------------

    def show(self):
        from collections import deque
        
        q = deque()
        q.append(self.root)
        level = 0
        levelss = []
        
        while q:
            size = len(q)
            level_nodes = []
            for i in range(size):
                n = q.popleft()
                if n.is_leaf:
                    level_nodes.append(f"{n.keys}")
                else:
                    level_nodes.append(f"{n.keys}")
                    inode: InternalNode = n  
                    for c in inode.children:
                        q.append(c)
            levelss.append(f"Level {level}: " + " | ".join(level_nodes))
            level += 1

        print("\n".join(levelss))
        print("-" * 60)

# TASK 1

In [2]:
def d1_sequence():
    
    tree = BPlusTree(d=1)

    L1 = LeafNode(); L1.keys = [13]
    L2 = LeafNode(); L2.keys = [16,19]
    L3 = LeafNode(); L3.keys = [20,25]

    root = InternalNode()
    root.keys = [16,20]

    root.children = [L1, L2, L3] 
    L1.parent = root; L2.parent = root; L3.parent = root 

    tree.init_tree_from_structure(root)
    print("Initial tree")
    tree.show()

    print("INSERT 27\n")
    tree.insert(27)
    tree.show()

    print("INSERT 14\n")
    tree.insert(14)
    tree.show()

    print("INSERT 22\n")
    tree.insert(22)
    tree.show()

    print("INSERT 21\n")
    tree.insert(21)
    tree.show()

    print("DELETE 21\n")
    tree.delete(21)
    tree.show()

    print("DELETE 22\n")
    tree.delete(22)
    tree.show()

    print("DELETE 20\n")
    tree.delete(20)
    tree.show()

    print("DELETE 25\n")
    tree.delete(25)
    tree.show()


if __name__ == "__main__":
    d1_sequence()


Initial tree
Level 0: [16, 20]
Level 1: [13] | [16, 19] | [20, 25]
------------------------------------------------------------
INSERT 27

Level 0: [20]
Level 1: [16] | [25]
Level 2: [13] | [16, 19] | [20] | [25, 27]
------------------------------------------------------------
INSERT 14

Level 0: [20]
Level 1: [16] | [25]
Level 2: [13, 14] | [16, 19] | [20] | [25, 27]
------------------------------------------------------------
INSERT 22

Level 0: [20]
Level 1: [16] | [25]
Level 2: [13, 14] | [16, 19] | [20, 22] | [25, 27]
------------------------------------------------------------
INSERT 21

Level 0: [20]
Level 1: [16] | [21, 25]
Level 2: [13, 14] | [16, 19] | [20] | [21, 22] | [25, 27]
------------------------------------------------------------
DELETE 21

Level 0: [20]
Level 1: [16] | [22, 25]
Level 2: [13, 14] | [16, 19] | [20] | [22] | [25, 27]
------------------------------------------------------------
DELETE 22

Level 0: [20]
Level 1: [16] | [22, 27]
Level 2: [13, 14] | [16, 1

# TASK 2,3

In [3]:
def d2_sequence():
    
    tree = BPlusTree(d=2)
    print("Initial tree")
    tree.show()

    for k in range(1, 21):          
        print(f"INSERT {k}\n")
        tree.insert(k)
        tree.show()

if __name__ == "__main__":
    d2_sequence()


Initial tree
Level 0: []
------------------------------------------------------------
INSERT 1

Level 0: [1]
------------------------------------------------------------
INSERT 2

Level 0: [1, 2]
------------------------------------------------------------
INSERT 3

Level 0: [1, 2, 3]
------------------------------------------------------------
INSERT 4

Level 0: [1, 2, 3, 4]
------------------------------------------------------------
INSERT 5

Level 0: [3]
Level 1: [1, 2] | [3, 4, 5]
------------------------------------------------------------
INSERT 6

Level 0: [3]
Level 1: [1, 2] | [3, 4, 5, 6]
------------------------------------------------------------
INSERT 7

Level 0: [3, 5]
Level 1: [1, 2] | [3, 4] | [5, 6, 7]
------------------------------------------------------------
INSERT 8

Level 0: [3, 5]
Level 1: [1, 2] | [3, 4] | [5, 6, 7, 8]
------------------------------------------------------------
INSERT 9

Level 0: [3, 5, 7]
Level 1: [1, 2] | [3, 4] | [5, 6] | [7, 8, 9]
-------